# Session 2 · Live on Jarvis: prepare the data and learn while it runs

One notebook for the whole session:
1. **Setup**: keys, packages, checks (run once per machine).
2. **Start** the data run in the background. It keeps going even if you close this tab.
3. **Watch it live**: files done, tokens so far, speed, time left, CPU and disk.
4. **Learn from the real data as it lands**: tokenizer lens, the newest shard, token frequencies, the model's insides.
5. **Gate, log, pause the machine.**

Re-opening later? Run cell 1, then jump to any section. Section 2 won't start a second copy of the run.

In [ ]:
# 1. Setup (safe to re-run)
%matplotlib inline
import os, sys, subprocess, getpass, glob, json, time, shutil
from pathlib import Path
REPO = Path.home() / 'Chotu'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/aditya950820/Chotu.git', str(REPO)], check=True)
os.chdir(REPO); sys.path.insert(0, str(REPO))
subprocess.run(['git', 'pull', '--rebase', '-q'])
STORAGE = Path.home() / 'chotu-data'; STORAGE.mkdir(exist_ok=True)
LOG, PIDF = STORAGE / 'run.log', STORAGE / 'run.pid'

# keys: create .env once, typed in hidden fields (never printed, never committed)
env = REPO / '.env'
if not env.exists():
    vals = {k: getpass.getpass(f'{k}: ') for k in ['HF_TOKEN', 'WANDB_API_KEY', 'JL_API_KEY', 'JL_MACHINE_ID']}
    env.write_text('\n'.join(f'{k}={v}' for k, v in vals.items()) + '\nWANDB_ENTITY=adityaagrawal-student\nWANDB_PROJECT=chotu\n')
    env.chmod(0o600)
for k in ['user.name', 'user.email']:
    if not subprocess.run(['git', 'config', k], capture_output=True, text=True).stdout.strip():
        subprocess.run(['git', 'config', k, input(f'git {k}: ')])

!grep -vE '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt jarvislabs
!python scripts/check_setup.py
!python scripts/stop_machine.py --dry-run
print(f'CPU cores: {os.cpu_count()}  ->  data run roughly {28/os.cpu_count():.1f} h of tokenizing (plus download)')
print(f'disk free in {STORAGE}: {shutil.disk_usage(STORAGE).free/1e9:.0f} GB (need ~25)')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU (fine for this session)'

## 2 · Start the data run (background)

First a 1-minute test, then the full run as a background process. It writes one train and one val shard per source file (14 files), and it's resumable: if it dies, run this cell again and finished files are skipped.

In [ ]:
def running():
    if not PIDF.exists(): return False
    try: os.kill(int(PIDF.read_text()), 0); return True
    except (OSError, ValueError): return False

if running():
    print('Already running (pid', PIDF.read_text(), '): not starting another copy.')
else:
    t = subprocess.run([sys.executable, 'scripts/prepare_data.py', '--storage', str(STORAGE), '--max-docs', '2000', '--max-files', '2'],
                       capture_output=True, text=True)
    print('\n'.join(l for l in t.stdout.splitlines() if l.startswith('[')))
    assert t.returncode == 0, t.stderr[-2000:]
    p = subprocess.Popen(f'nohup {sys.executable} scripts/prepare_data.py --storage {STORAGE} >> {LOG} 2>&1 &  echo $!',
                         shell=True, stdout=subprocess.PIPE, text=True, cwd=REPO)
    PIDF.write_text(p.stdout.read().strip())
    print('Full run started, pid', PIDF.read_text(), '| log:', LOG)

## 3 · Watch it live

Refreshes every 30 s for as long as you let it run. Stopping this cell (■) only stops *watching*; the data run keeps going.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
history = []
def status():
    done = [json.load(open(f)) for f in sorted(glob.glob(str(STORAGE / 'shards' / '*.done')))]
    bins = glob.glob(str(STORAGE / 'shards' / '*.bin')) + glob.glob(str(STORAGE / 'shards' / '*.bin.tmp'))
    tokens = sum(os.path.getsize(b) for b in bins) // 2
    return len(done), tokens, done

try:
    while True:
        n, tokens, done = status(); now = time.time(); history.append((now, tokens))
        clear_output(wait=True)
        rate = (history[-1][1] - history[0][1]) / max(1, history[-1][0] - history[0][0]) if len(history) > 1 else 0
        target = 10.0e9
        eta = (target - tokens) / rate / 3600 if rate > 0 else float('nan')
        print(f'running: {running()} | files done: {n}/14 | tokens on disk: {tokens/1e9:.2f}B of ~10B')
        print(f'speed (this view): {rate/1e6:.2f}M tokens/s | rough time left: {eta:.1f} h')
        print(f'CPU load (1 min): {os.getloadavg()[0]:.1f} of {os.cpu_count()} cores | disk free: {shutil.disk_usage(STORAGE).free/1e9:.0f} GB')
        print('--- last log lines ---'); print(''.join(open(LOG).readlines()[-4:]) if LOG.exists() else '(no log yet)')
        if len(history) > 1:
            t0 = history[0][0]
            plt.figure(figsize=(8, 2.5)); plt.plot([(t - t0)/60 for t, _ in history], [k/1e9 for _, k in history], marker='.')
            plt.xlabel('minutes watching'); plt.ylabel('B tokens'); plt.title('Tokens written so far'); plt.show()
        if not running() and n == 14: print('Finished: go to section 5.'); break
        time.sleep(30)
except KeyboardInterrupt:
    print('Stopped watching. The data run is still going:', running())

## 4 · Learn from the real data while it runs

These use the newest *finished* shard, so re-run them later to see more data. Answer the questions before reading on; write answers in `docs/LEARNING.md`, or ask Claude: *"Teach me Socratically, one question at a time, diagrams over maths. Topic: …"*

In [ ]:
from IPython.display import HTML, display
from transformers import AutoTokenizer
from chotu.config import TOKENIZER_ID, chotu_config
from chotu import xray as X
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID)
for text in ["The cat sat on the mat.", "In 2048, Chotu learned that photosynthesis isn't magic.",
             "antidisestablishmentarianism", "Namaste! मेरा नाम चोटू है"]:
    display(HTML(X.token_lens(tok, text)))

**4a · Tokenizer lens.** Why is "2048" four tokens but "cat" one? Why does Hindi cost so many more tokens than English, and what does that mean for a model trained on English text?

In [ ]:
finished = sorted(glob.glob(str(STORAGE / 'shards' / 'train_*.bin'))) or sorted(glob.glob(str(STORAGE / 'test' / 'shards' / 'train_*.bin')))
print(f'{len(finished)} finished shard(s); newest: {finished[-1]}')
for seed in [1, 2]:
    display(HTML(X.shard_window(tok, finished[-1], n=120, seed=seed)))

**4b · Shard microscope.** This is literally what the model will read: numbers, with a red end-of-document marker between documents. Why does it need that marker?

In [ ]:
counts, total = X.token_frequency(finished, max_tokens=100_000_000)
fig = X.plot_token_frequency(tok, counts, total)

**4c · Token frequency** (re-run as more shards finish; the curve barely changes, and that's a lesson too). The top 10 tokens are about a quarter of all text. If the model only learned those, how good would it be? How often will it see a rare token in 10B?

In [ ]:
import torch
from transformers import Qwen3ForCausalLM
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
model = Qwen3ForCausalLM(chotu_config(attn_implementation='eager')).to(device)
fig = X.plot_param_map(model.cpu()); model.to(device)
x = X.xray(model.cpu(), tok, "The capital of France is Paris, and the capital of Italy is")
for t, p in x['next'][:5]: print(f'{t!r:16s} {p:.4%}')
print(f'chance = {1/49152:.4%}')
fig = X.plot_attention(x, layer=0)
fig = X.plot_logit_lens(x, every=2)
fig = X.plot_layer_signal(x)

**4d · Inside the untrained model.** Why are attention maps triangles? Why does the embedding row of the logit lens predict the *current* word? After training, at which layers would you expect "Rome" to appear? Keep these pictures for the before/after comparison in Session 3.

## 5 · Gate, log, pause

Gate: about **10B train tokens**, a validation split, and the 10 report samples read as clean English.

In [ ]:
from IPython.display import Markdown
Markdown((STORAGE / 'reports' / 'data_report.md').read_text())

In [ ]:
# Add your entry at the top of the log in docs/PROGRESS.md (numbers from the report), then:
!git add docs/ && git commit -m "Session 2 on Jarvis: data prepared" && git pull --rebase -q && git push

In [ ]:
# Pause the machine (files and packages are kept; storage is still billed). Only when the data run is finished!
assert not running(), 'Data run still going: wait for it to finish'
!python scripts/stop_machine.py